## 1: Import library

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings

##  2: settings

In [3]:
# datasets
AMAZON_PATH = r'D:\University\SBU\Data Mining\project\code\datasets\Movies_and_TV.csv'
RATINGS_PATH = r'D:\University\SBU\Data Mining\project\code\datasets\Ratings.csv'

# data setting
K_CORE = 5 # keep users with at least K_CORE interaction
n_topusers = 1500 # top active users
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Federated Learning
GLOBAL_ROUNDS = 40        # num of rounds to update
LOCAL_EPOCHS = 5          # each domain how many epochs train on its local data in each federal
WARMUP_ROUNDS = 5         # rounds without Sinkhorn in first epoches

# model setting
N_FACTORS = 64            # embedding dimension
LEARNING_RATE = 0.005
GAMMA = 0.1               # weight for Sinkhorn
EPSILON = 0.1             # epsilon for Sinkhorn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"   Device: {DEVICE}")
print(f"   Global rounds: {GLOBAL_ROUNDS}")

   Device: cuda
   Global rounds: 40


## 3: Federated Implementation

In [4]:
# load py
%run federated_no3_cdr.py
%run data_loader.py

## 4: load datasets

In [5]:
# load datasets
df_amazon = load_amazon_csv(AMAZON_PATH)
df_books = load_book_crossing_csv(RATINGS_PATH)

top_amz_users = df_amazon['user_id'].value_counts().nlargest(n_topusers).index
df_amazon = df_amazon[df_amazon['user_id'].isin(top_amz_users)]

top_bks_users = df_books['user_id'].value_counts().nlargest(n_topusers).index
df_books = df_books[df_books['user_id'].isin(top_bks_users)]

df_amazon = filter_k_core(df_amazon, k=5)
df_books = filter_k_core(df_books, k=5)


print(f"\nAmazon Ready: {len(df_amazon):,} rows")
print(f"Books Ready:  {len(df_books):,} rows")


Loading Movies_and_TV.csv...
   1,048,576 rows read
   1,048,576 valid interactions loaded
   Rating range: [1.0, 5.0]
Loading D:\University\SBU\Data Mining\project\code\datasets\Ratings.csv...
   1,149,780 rows read
   Columns: ['User-ID', 'ISBN', 'Rating']
   Identified: user='User-ID', item='ISBN', rating='Rating'
   Converting rating to numeric...
   433,671 valid interactions loaded
   Rating range: [1.0, 10.0]
Filtering 5-core...
   71,864 interactions remaining (6.5% removed)
   Users: 1,500, Items: 3,491
Filtering 5-core...
   51,410 interactions remaining (71.9% removed)
   Users: 1,365, Items: 5,103

Amazon Ready: 71,864 rows
Books Ready:  51,410 rows


##  5: Federated Learning Data preparing

In [6]:
# Federated Learning preparing
# Mapping
user_map_amazon, item_map_amazon = create_user_item_mappings(df_amazon, item_col='product_id')
user_map_books, item_map_books = create_user_item_mappings(df_books, item_col='product_id')

n_users_amazon = len(user_map_amazon)
n_items_amazon = len(item_map_amazon)
n_users_books = len(user_map_books)
n_items_books = len(item_map_books)

print(f" Amazon: {n_users_amazon:,} users × {n_items_amazon:,} items")
print(f" Books: {n_users_books:,} users × {n_items_books:,} items")

# convert into array, control int or float, remove usless columns
data_amazon = preprocess_data(df_amazon, user_map_amazon, item_map_amazon, item_col='product_id')
data_books = preprocess_data(df_books, user_map_books, item_map_books, item_col='product_id')

# split train/test
train_amazon, test_amazon = train_test_split(data_amazon, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_books, test_books = train_test_split(data_books, test_size=TEST_SIZE, random_state=RANDOM_STATE)


 Amazon: 1,500 users × 3,491 items
 Books: 1,365 users × 5,103 items


##  6:  Centralized Learning

In [8]:
from no3_cdr_implementation import SNO3_CDR, MatrixFactorization, rmse, mae

# model
centralized = SNO3_CDR(
    model_class=MatrixFactorization,
    n_users_d1=n_users_amazon,
    n_users_d2=n_users_books,
    n_items_d1=n_items_amazon,
    n_items_d2=n_items_books,
    n_factors=N_FACTORS,
    gamma=GAMMA,
    epsilon=EPSILON,
    device=DEVICE
)

# Train model
centralized.train(
    train_amazon, 
    train_books,
    epochs=GLOBAL_ROUNDS,
    lr=LEARNING_RATE,
    warmup_epochs=WARMUP_ROUNDS,
    task='rating'
)

# evaluation
print("\n evaluation Centralized:")
pred_amazon_cent = centralized.predict(test_amazon[:, 0], test_amazon[:, 1], domain=1)
pred_books_cent = centralized.predict(test_books[:, 0], test_books[:, 1], domain=2)

true_amazon = test_amazon[:, 2]
true_books = test_books[:, 2]

rmse_amazon_cent = rmse(pred_amazon_cent, true_amazon)
mae_amazon_cent = mae(pred_amazon_cent, true_amazon)

rmse_books_cent = rmse(pred_books_cent, true_books)
mae_books_cent = mae(pred_books_cent, true_books)

print(f"   Amazon - RMSE: {rmse_amazon_cent:.4f}, MAE: {mae_amazon_cent:.4f}")
print(f"   Books  - RMSE: {rmse_books_cent:.4f}, MAE: {mae_books_cent:.4f}")
print(f"   Average- RMSE: {(rmse_amazon_cent+rmse_books_cent)/2:.4f}")


Warmup Epoch 5/40, Loss: 133.3915
Epoch 10/40, Total Loss: 98.0184, Rec Loss: 97.6191, Sink Loss: 3.9940
Epoch 15/40, Total Loss: 69.9868, Rec Loss: 69.3617, Sink Loss: 6.2508
Epoch 20/40, Total Loss: 55.8047, Rec Loss: 54.7030, Sink Loss: 11.0171
Epoch 25/40, Total Loss: 48.8608, Rec Loss: 46.4688, Sink Loss: 23.9201
Epoch 30/40, Total Loss: 47.9440, Rec Loss: 42.8602, Sink Loss: 50.8379
Epoch 35/40, Total Loss: 48.7423, Rec Loss: 40.8671, Sink Loss: 78.7522
Epoch 40/40, Total Loss: 51.0083, Rec Loss: 39.6177, Sink Loss: 113.9052

 evaluation Centralized:
   Amazon - RMSE: 0.8819, MAE: 0.6226
   Books  - RMSE: 1.6300, MAE: 1.2660
   Average- RMSE: 1.2559


## 7: Federated Learning

In [9]:
# Federated Learning

# Federated model
federated = FederatedNO3_CDR(
    n_users_d1=n_users_amazon,
    n_users_d2=n_users_books,
    n_items_d1=n_items_amazon,
    n_items_d2=n_items_books,
    n_factors=N_FACTORS,
    gamma=GAMMA,
    epsilon=EPSILON,
    device=DEVICE
)

# train
federated.train_federated(
    train_amazon,
    train_books,
    global_rounds=GLOBAL_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    lr=LEARNING_RATE,
    warmup_rounds=WARMUP_ROUNDS
)


# evaluation
print("\n evaluation Federated:")

# set model in evaluation mode
federated.client_d1.model.eval()
federated.client_d2.model.eval()

# prepare data to predict with GPU
with torch.no_grad():
    # Client 1
    users_amz = torch.tensor(test_amazon[:, 0], dtype=torch.long, device=DEVICE)
    items_amz = torch.tensor(test_amazon[:, 1], dtype=torch.long, device=DEVICE)
    pred_amazon_fed = federated.client_d1.model(users_amz, items_amz).cpu().numpy()

    # Client 2
    users_bks = torch.tensor(test_books[:, 0], dtype=torch.long, device=DEVICE)
    items_bks = torch.tensor(test_books[:, 1], dtype=torch.long, device=DEVICE)
    pred_books_fed = federated.client_d2.model(users_bks, items_bks).cpu().numpy()

true_amazon = test_amazon[:, 2]
true_books = test_books[:, 2]

rmse_amazon_fed = rmse(pred_amazon_fed, true_amazon)
mae_amazon_fed = mae(pred_amazon_fed, true_amazon)

rmse_books_fed = rmse(pred_books_fed, true_books)
mae_books_fed = mae(pred_books_fed, true_books)

print(f"   Amazon (Fed) - RMSE: {rmse_amazon_fed:.4f}, MAE: {mae_amazon_fed:.4f}")
print(f"   Books  (Fed) - RMSE: {rmse_books_fed:.4f}, MAE: {mae_books_fed:.4f}")
print(f"   Average (Fed)- RMSE: {(rmse_amazon_fed + rmse_books_fed) / 2:.4f}")

results_fed = {
    'domain1': {'rmse': rmse_amazon_fed, 'mae': mae_amazon_fed},
    'domain2': {'rmse': rmse_books_fed, 'mae': mae_books_fed},
    'average': {
        'rmse': (rmse_amazon_fed + rmse_books_fed) / 2,
        'mae': (mae_amazon_fed + mae_books_fed) / 2
    }
}


Federated Server initialized (coordinator mode)
Client Domain-1 initialized: 1500 users, 3491 items
Client Domain-2 initialized: 1365 users, 5103 items
Federated SNO3-CDR initialized (Gamma: 0.1, Epsilon: 0.1)
Starting Federated Training (SNO3)
Global rounds: 40, Warmup: 5
Round 1 | D1 Loss: 3.2846 | D2 Loss: 19.9599
Round 2 | D1 Loss: 0.4838 | D2 Loss: 1.7445
Round 3 | D1 Loss: 0.2656 | D2 Loss: 0.9761
Round 4 | D1 Loss: 0.2043 | D2 Loss: 0.5731
Round 5 | D1 Loss: 0.1923 | D2 Loss: 0.3872
Round 6 | D1 Loss: 0.1873 | D2 Loss: 0.2988 | Sinkhorn: 0.5914
Round 7 | D1 Loss: 0.1841 | D2 Loss: 0.2606 | Sinkhorn: 0.7869
Round 8 | D1 Loss: 0.1839 | D2 Loss: 0.2528 | Sinkhorn: 0.8184
Round 9 | D1 Loss: 0.1826 | D2 Loss: 0.2455 | Sinkhorn: 0.8286
Round 10 | D1 Loss: 0.1807 | D2 Loss: 0.2454 | Sinkhorn: 0.8218
Round 11 | D1 Loss: 0.1805 | D2 Loss: 0.2441 | Sinkhorn: 0.8397
Round 12 | D1 Loss: 0.1799 | D2 Loss: 0.2415 | Sinkhorn: 0.8337
Round 13 | D1 Loss: 0.1812 | D2 Loss: 0.2417 | Sinkhorn: 0.83

##  8: Comparison

In [10]:
# Comparison

# table
comparison_data = {
    'Method': ['Centralized', 'Federated'],
    'Amazon RMSE': [
        rmse_amazon_cent,
        results_fed['domain1']['rmse']
    ],
    'Books RMSE': [
        rmse_books_cent,
        results_fed['domain2']['rmse']
    ],
    'Average RMSE': [
        (rmse_amazon_cent + rmse_books_cent) / 2,
        results_fed['average']['rmse']
    ],
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

,Method,Amazon RMSE,Books RMSE,Average RMSE
0,Centralized,0.881879,1.629967,1.255923
1,Federated,0.880839,1.527642,1.204240
